# 0_1_5 GNN Cross-Modal Scoring

This notebook runs **after** `0_1` (traditional rotation scoring) and **before** `0_2` (Powell weight optimization).

It adds **GNN_Cosine** as the 9th metric to the rotation scores CSV.

## Pipeline
1. Train GNN model(s) using 4-fold CV (for fair scoring) and/or full training (for final model)
2. Generate GNN cosine similarity scores for each (gene, sample, angle) combination
3. Merge GNN scores with the traditional metrics CSV from `0_1`
4. Output `rotation_scores_with_gnn.csv` for `0_2` to consume

## Rotation Convention
**CRITICAL**: This notebook rotates **RNA** (not MSI) to match baseline's `0_1` convention.

In [2]:
# ============================================================================
# MODE & DATA CONFIGURATION
# ============================================================================

# TRAIN_MODE options:
#   "CV_AND_FULL"  - (Default) 4-fold CV for fair scoring + full model training
#   "CV_ONLY"      - Only 4-fold CV scoring (for Powell weight optimization)
#   "FULL_ONLY"    - Only train final model on all 16 samples
#   "INFERENCE"    - Load existing model, skip all training
TRAIN_MODE = "CV_AND_FULL"

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import time
import json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import softmax as pyg_softmax
from sklearn.neighbors import NearestNeighbors
import scanpy as sc

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# === Paths ===
DATA_DIR = "/home/ajarrah/PhD_Thesis/chapter_4/dummy_data_50"
INPUT_CSV = "./rotation_analysis_optimized_50_340_20_2/rotation_scores_optimized.csv"  # From 0_1
OUTPUT_DIR = "./gnn_integration_output"
OUTPUT_CSV = "./rotation_scores_with_gnn.csv"  # For 0_2 to read
FINAL_MODEL_PATH = os.path.join(OUTPUT_DIR, "final_model.pth")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# === 50 Target Patterns ===
TARGET_GENES = [
    # Gradients (6)
    "Gene_Gradient_X", "Gene_Gradient_Y",
    "Gene_Gradient_Diagonal_NE", "Gene_Gradient_Diagonal_NW",
    "Gene_Gradient_Radial_In", "Gene_Gradient_Radial_Out",
    # Waves & Stripes (8)
    "Gene_Stripes_Vertical", "Gene_Stripes_Horizontal",
    "Gene_Stripes_Diagonal_45", "Gene_Stripes_Diagonal_135",
    "Gene_Waves_Concentric", "Gene_Waves_Spiral",
    "Gene_Waves_Interference", "Gene_Waves_Ripple",
    # Blobs & Spots (10)
    "Gene_Blob_Center", "Gene_Blob_TopRight", "Gene_Blob_TopLeft",
    "Gene_Blob_BottomRight", "Gene_Blob_BottomLeft",
    "Gene_Spots_Grid_Dense", "Gene_Spots_Grid_Sparse",
    "Gene_Spots_Random_Large", "Gene_Spots_Triangular", "Gene_Spots_Hexagonal",
    # Rings & Donuts (6)
    "Gene_Ring_Inner", "Gene_Ring_Outer", "Gene_Ring_Double",
    "Gene_Ring_Eccentric", "Gene_Ring_Elliptical", "Gene_Ring_Partial",
    # Geometric (8)
    "Gene_Checkerboard_Fine", "Gene_Checkerboard_Coarse",
    "Gene_Quadrant_Alternating", "Gene_Sectors_4", "Gene_Sectors_8",
    "Gene_Triangle_Pattern", "Gene_Diamond_Pattern", "Gene_Honeycomb",
    # Complex Biological-like (12)
    "Gene_Cortical_Layers", "Gene_Hotspot_Cluster", "Gene_Edge_Enhancement",
    "Gene_Core_Shell", "Gene_Branching", "Gene_Laminar_Curved",
    "Gene_Mosaic_Irregular", "Gene_Gradient_Sigmoid", "Gene_Bimodal_Distribution",
    "Gene_Punctate_Dense", "Gene_Periventricular", "Gene_Asymmetric_Lobe",
]

TARGET_MZS = [gene.replace("Gene_", "MZ_") for gene in TARGET_GENES]
GROUND_TRUTH = {gene: gene.replace("Gene_", "MZ_") for gene in TARGET_GENES}

ALL_SAMPLES = ['YC_1', 'YC_2', 'YC_3', 'YC_4',
               'AC_1', 'AC_2', 'AC_3', 'AC_4',
               'YAD_1', 'YAD_2', 'YAD_3', 'YAD_4',
               'AAD_1', 'AAD_2', 'AAD_3', 'AAD_4']

MSI_FILES = {
    'YC_1': 'halfbrain_yc_1_filtered_common.h5ad',
    'YC_2': 'halfbrain_yc_2_filtered_common.h5ad',
    'YC_3': 'halfbrain_yc_3_filtered_common.h5ad',
    'YC_4': 'halfbrain_yc_4_filtered_common.h5ad',
    'AC_1': 'halfbrain_ac_1_filtered_common.h5ad',
    'AC_2': 'halfbrain_ac_2_filtered_common.h5ad',
    'AC_3': 'halfbrain_ac_3_filtered_common.h5ad',
    'AC_4': 'halfbrain_ac_4_filtered_common.h5ad',
    'YAD_1': 'halfbrain_yad_1_filtered_common.h5ad',
    'YAD_2': 'halfbrain_yad_2_filtered_common.h5ad',
    'YAD_3': 'halfbrain_yad_3_filtered_common.h5ad',
    'YAD_4': 'halfbrain_yad_4_filtered_common.h5ad',
    'AAD_1': 'halfbrain_aad_1_filtered_common.h5ad',
    'AAD_2': 'halfbrain_aad_2_filtered_common.h5ad',
    'AAD_3': 'halfbrain_aad_3_filtered_common.h5ad',
    'AAD_4': 'halfbrain_aad_4_filtered_common.h5ad',
}

# 4-Fold CV: each fold holds out one sample number (1/2/3/4) across all groups
CV_FOLDS = {
    1: {'test': ['YC_1', 'AC_1', 'YAD_1', 'AAD_1'],
        'train': ['YC_2','YC_3','YC_4','AC_2','AC_3','AC_4','YAD_2','YAD_3','YAD_4','AAD_2','AAD_3','AAD_4']},
    2: {'test': ['YC_2', 'AC_2', 'YAD_2', 'AAD_2'],
        'train': ['YC_1','YC_3','YC_4','AC_1','AC_3','AC_4','YAD_1','YAD_3','YAD_4','AAD_1','AAD_3','AAD_4']},
    3: {'test': ['YC_3', 'AC_3', 'YAD_3', 'AAD_3'],
        'train': ['YC_1','YC_2','YC_4','AC_1','AC_2','AC_4','YAD_1','YAD_2','YAD_4','AAD_1','AAD_2','AAD_4']},
    4: {'test': ['YC_4', 'AC_4', 'YAD_4', 'AAD_4'],
        'train': ['YC_1','YC_2','YC_3','AC_1','AC_2','AC_3','YAD_1','YAD_2','YAD_3','AAD_1','AAD_2','AAD_3']},
}

# GNN hyperparameters (from v5)
CONFIG = {
    'hidden_dim': 128,
    'embedding_dim': 64,
    'projection_dim': 64,
    'num_heads': 4,
    'num_layers': 3,
    'dropout': 0.1,
    'learning_rate': 0.0005,
    'weight_decay': 1e-4,
    'epochs': 400,
    'patience': 60,
    'batch_size': 50,
    'temperature': 0.07,
    'k_neighbors': 8,
    'aug_rotation': True,
    'aug_flip': True,
    'aug_rna_dropout': 0.2,
    'aug_msi_noise': 0.05,
    'aug_coord_jitter': 0.02,
    'aug_rotation_offset_deg': 0,       # CHANGED: was 20, removed to improve accuracy
    'noise_negatives_per_batch': 50,     # Noise MZ negatives per training batch
    'noise_sub_batch_size': 10,          # Encode noise in sub-batches to avoid OOM
}

# Rotation angles matching baseline's 0_1: 340-360 (= -20 to 0) then 0-20, step 2
ROTATION_ANGLES_DEG = list(range(-20, 22, 2))  # [-20, -18, ..., 18, 20]
# For CSV output, baseline uses 340-360 convention for negative angles
def angle_to_csv(a):
    """Convert signed angle to baseline's CSV convention (340 = -20)."""
    return a if a >= 0 else 360 + a

print(f"TRAIN_MODE: {TRAIN_MODE}")
print(f"Rotation angles: {ROTATION_ANGLES_DEG}")
print(f"CV folds: {len(CV_FOLDS)}")
print(f"Samples: {len(ALL_SAMPLES)}")
print(f"Noise negatives per batch: {CONFIG['noise_negatives_per_batch']}")
print(f"Noise sub-batch size: {CONFIG['noise_sub_batch_size']}")
print(f"Rotation offset aug: {CONFIG['aug_rotation_offset_deg']}°")

Using device: cuda
TRAIN_MODE: CV_AND_FULL
Rotation angles: [-20, -18, -16, -14, -12, -10, -8, -6, -4, -2, 0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20]
CV folds: 4
Samples: 16
Noise negatives per batch: 50
Noise sub-batch size: 10
Rotation offset aug: 0°


## Model Architecture

Self-contained GNN model classes (copied from `gnn_crossmodal_v5.py` to avoid import issues).

In [3]:
# ============================================================================
# GNN MODEL (from gnn_crossmodal_v5.py)
# ============================================================================

class SpatialEncoder(nn.Module):
    """Input Proj -> GATv2 with Residual + LayerNorm -> Attention Pooling -> Output Proj"""

    def __init__(self, input_dim=3, hidden_dim=128, embedding_dim=64,
                 num_heads=4, num_layers=3, dropout=0.1):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(
                GATv2Conv(hidden_dim, hidden_dim // num_heads,
                         heads=num_heads, dropout=dropout, concat=True)
            )
            self.norms.append(nn.LayerNorm(hidden_dim))

        self.pool_attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 1)
        )

        self.output_proj = nn.Sequential(
            nn.Linear(hidden_dim, embedding_dim),
            nn.LayerNorm(embedding_dim)
        )

    def forward(self, x, edge_index):
        h = self.input_proj(x)
        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h, edge_index)
            h_new = norm(h_new)
            h_new = F.gelu(h_new)
            h = h + h_new  # Residual
        attn_weights = self.pool_attn(h)
        attn_weights = F.softmax(attn_weights, dim=0)
        global_emb = (h * attn_weights).sum(dim=0)
        return self.output_proj(global_emb)

    def encode_batch(self, batch):
        """Batched forward pass with per-graph attention pooling."""
        h = self.input_proj(batch.x)
        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h, batch.edge_index)
            h_new = norm(h_new)
            h_new = F.gelu(h_new)
            h = h + h_new
        attn_scores = self.pool_attn(h).squeeze(-1)
        attn_weights = pyg_softmax(attn_scores, batch.batch)
        weighted = h * attn_weights.unsqueeze(-1)
        num_graphs = batch.batch.max().item() + 1
        global_embs = torch.zeros(num_graphs, h.size(-1), device=h.device)
        global_embs.scatter_add_(0, batch.batch.unsqueeze(-1).expand_as(weighted), weighted)
        return self.output_proj(global_embs)


class CrossModalGNN(nn.Module):
    """Two-tower cross-modal GNN."""

    def __init__(self, config):
        super().__init__()
        self.config = config
        self.rna_encoder = SpatialEncoder(
            input_dim=3, hidden_dim=config['hidden_dim'],
            embedding_dim=config['embedding_dim'], num_heads=config['num_heads'],
            num_layers=config['num_layers'], dropout=config['dropout']
        )
        self.msi_encoder = SpatialEncoder(
            input_dim=3, hidden_dim=config['hidden_dim'],
            embedding_dim=config['embedding_dim'], num_heads=config['num_heads'],
            num_layers=config['num_layers'], dropout=config['dropout']
        )
        self.rna_projection = nn.Sequential(
            nn.Linear(config['embedding_dim'], config['embedding_dim']),
            nn.GELU(),
            nn.Linear(config['embedding_dim'], config['projection_dim'])
        )
        self.msi_projection = nn.Sequential(
            nn.Linear(config['embedding_dim'], config['embedding_dim']),
            nn.GELU(),
            nn.Linear(config['embedding_dim'], config['projection_dim'])
        )

    def encode_rna(self, x, edge_index):
        emb = self.rna_encoder(x, edge_index)
        proj = self.rna_projection(emb)
        return F.normalize(proj, dim=-1)

    def encode_msi(self, x, edge_index):
        emb = self.msi_encoder(x, edge_index)
        proj = self.msi_projection(emb)
        return F.normalize(proj, dim=-1)


class InfoNCELoss(nn.Module):
    """Bidirectional InfoNCE loss with optional noise negatives."""

    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, rna_embs, msi_embs, noise_embs=None):
        batch_size = rna_embs.size(0)
        labels = torch.arange(batch_size, device=rna_embs.device)

        # RNA -> MSI direction: include noise as extra negatives
        if noise_embs is not None and len(noise_embs) > 0:
            all_msi = torch.cat([msi_embs, noise_embs], dim=0)  # (B+N) x D
        else:
            all_msi = msi_embs
        sim_r2m = torch.matmul(rna_embs, all_msi.T) / self.temperature  # B x (B+N)
        loss_r2m = F.cross_entropy(sim_r2m, labels)  # labels point to first B columns

        # MSI -> RNA direction: only positive MSI, no noise (noise has no RNA match)
        sim_m2r = torch.matmul(msi_embs, rna_embs.T) / self.temperature  # B x B
        loss_m2r = F.cross_entropy(sim_m2r, labels)

        return (loss_r2m + loss_m2r) / 2


print("Model classes defined.")

Model classes defined.


## Data Loading & Graph Utilities

In [4]:
# ============================================================================
# DATA LOADING & GRAPH CONSTRUCTION
# ============================================================================

def build_knn_graph(coords, k=8):
    """Build k-NN graph (bidirectional edges)."""
    nbrs = NearestNeighbors(n_neighbors=k + 1, algorithm='ball_tree').fit(coords)
    _, indices = nbrs.kneighbors(coords)
    edges = []
    for i in range(len(coords)):
        for j in indices[i, 1:]:
            edges.append([i, j])
            edges.append([j, i])
    return torch.tensor(edges, dtype=torch.long).t().contiguous()


def extract_pattern_data(adata, feature_name):
    """Extract coordinates and values for a feature."""
    if feature_name not in adata.var_names:
        return None, None
    idx = list(adata.var_names).index(feature_name)
    values = adata.X[:, idx]
    if hasattr(values, 'toarray'):
        values = values.toarray().flatten()
    else:
        values = np.asarray(values).flatten()
    coords = np.column_stack([
        adata.obs['x_um'].values.astype(float),
        adata.obs['y_um'].values.astype(float)
    ])
    return coords, values


def make_features(coords, values):
    """Build z-score normalized node features [value, x, y]."""
    values_norm = (values - values.mean()) / (values.std() + 1e-8)
    x_norm = (coords[:, 0] - coords[:, 0].mean()) / (coords[:, 0].std() + 1e-8)
    y_norm = (coords[:, 1] - coords[:, 1].mean()) / (coords[:, 1].std() + 1e-8)
    return torch.tensor(np.column_stack([values_norm, x_norm, y_norm]), dtype=torch.float32)


def prepare_graph(coords, values, k=8):
    """Create k-NN graph with z-score normalization."""
    x = make_features(coords, values)
    edge_index = build_knn_graph(coords, k)
    return Data(x=x, edge_index=edge_index)


# --- Load all data ---
print("Loading data...", flush=True)
rna_data = {}
msi_data = {}
for sample_id in ALL_SAMPLES:
    rna_path = os.path.join(DATA_DIR, 'rna', f'{sample_id}.h5ad')
    if os.path.exists(rna_path):
        rna_data[sample_id] = sc.read_h5ad(rna_path)
    msi_path = os.path.join(DATA_DIR, 'msi', MSI_FILES[sample_id])
    if os.path.exists(msi_path):
        msi_data[sample_id] = sc.read_h5ad(msi_path)

print(f"  Loaded {len(rna_data)} RNA samples, {len(msi_data)} MSI samples")
s0 = ALL_SAMPLES[0]
print(f"  RNA: {rna_data[s0].shape[1]} features, {rna_data[s0].shape[0]} spots/sample")
print(f"  MSI: {msi_data[s0].shape[1]} features, {msi_data[s0].shape[0]} spots/sample")

Loading data...
  Loaded 16 RNA samples, 16 MSI samples
  RNA: 550 features, 2333 spots/sample
  MSI: 550 features, 5625 spots/sample


## Training Functions

Includes data augmentation, dataset, and training loop.

In [5]:
# ============================================================================
# DATA AUGMENTATION & TRAINING
# ============================================================================

class SpatialAugmentation:
    """Augmentation with synchronized geometric + independent modality transforms."""

    def __init__(self, config):
        self.config = config

    def apply_geometric_transform(self, coords, seed=None):
        if seed is not None:
            np.random.seed(seed)
        coords = coords.copy()
        center = coords.mean(axis=0)
        coords = coords - center
        if self.config.get('aug_rotation', True):
            angle = np.random.uniform(0, 2 * np.pi)
            cos_a, sin_a = np.cos(angle), np.sin(angle)
            rot = np.array([[cos_a, -sin_a], [sin_a, cos_a]])
            coords = coords @ rot.T
        if self.config.get('aug_flip', True):
            if np.random.random() > 0.5:
                coords[:, 0] = -coords[:, 0]
            if np.random.random() > 0.5:
                coords[:, 1] = -coords[:, 1]
        jitter = self.config.get('aug_coord_jitter', 0.02)
        if jitter > 0:
            noise = np.random.normal(0, jitter * coords.std(), coords.shape)
            coords = coords + noise
        coords = coords + center
        return coords

    def apply_relative_rotation(self, coords, max_offset_deg):
        """Small additional rotation for cross-modal misalignment."""
        coords = coords.copy()
        center = coords.mean(axis=0)
        coords = coords - center
        offset_rad = np.deg2rad(np.random.uniform(-max_offset_deg, max_offset_deg))
        cos_o, sin_o = np.cos(offset_rad), np.sin(offset_rad)
        rot = np.array([[cos_o, -sin_o], [sin_o, cos_o]])
        coords = coords @ rot.T
        coords = coords + center
        return coords

    def augment_pair(self, rna_coords, rna_values, msi_coords, msi_values, augment=True):
        if not augment:
            return rna_coords, rna_values, msi_coords, msi_values
        # Synchronized geometric transform
        geo_seed = np.random.randint(0, 100000)
        rna_coords = self.apply_geometric_transform(rna_coords, seed=geo_seed)
        msi_coords = self.apply_geometric_transform(msi_coords, seed=geo_seed)
        # Cross-modal relative rotation offset (MSI only)
        max_offset = self.config.get('aug_rotation_offset_deg', 0)
        if max_offset > 0:
            msi_coords = self.apply_relative_rotation(msi_coords, max_offset)
        # Independent modality augmentation
        rna_values = rna_values.copy()
        dropout_rate = self.config.get('aug_rna_dropout', 0.2)
        if dropout_rate > 0:
            nonzero_mask = rna_values > 0
            dropout_mask = np.random.random(len(rna_values)) < dropout_rate
            rna_values[nonzero_mask & dropout_mask] = 0
        msi_values = msi_values.copy()
        noise_level = self.config.get('aug_msi_noise', 0.05)
        if noise_level > 0:
            noise = np.random.normal(0, noise_level * msi_values.std(), len(msi_values))
            msi_values = msi_values + noise
        return rna_coords, rna_values, msi_coords, msi_values


class PrecomputedDataset:
    """Dataset with pre-computed k-NN graphs and noise MZ negatives."""

    def __init__(self, rna_data, msi_data, samples, config):
        self.config = config
        self.augmentor = SpatialAugmentation(config)
        self.pairs = []

        # --- Pre-compute positive pairs (same as before) ---
        count = 0
        for sample_id in samples:
            if sample_id not in rna_data or sample_id not in msi_data:
                continue
            rna = rna_data[sample_id]
            msi = msi_data[sample_id]
            for gene, mz in GROUND_TRUTH.items():
                rna_coords, rna_values = extract_pattern_data(rna, gene)
                msi_coords, msi_values = extract_pattern_data(msi, mz)
                if rna_coords is not None and msi_coords is not None:
                    count += 1
                    rna_edge_index = build_knn_graph(rna_coords, config['k_neighbors'])
                    msi_edge_index = build_knn_graph(msi_coords, config['k_neighbors'])
                    self.pairs.append({
                        'sample_id': sample_id, 'gene': gene, 'mz': mz,
                        'rna_coords': rna_coords.copy(), 'rna_values': rna_values.copy(),
                        'msi_coords': msi_coords.copy(), 'msi_values': msi_values.copy(),
                        'rna_edge_index': rna_edge_index, 'msi_edge_index': msi_edge_index,
                    })
                    if count % 100 == 0:
                        print(f"    Pre-computed {count} graph pairs...", flush=True)
        print(f"  Dataset: {len(self.pairs)} positive pairs from {len(samples)} samples", flush=True)

        # --- Pre-compute noise MZ data per sample ---
        # All MZ features in the same sample share coordinates and edge_index.
        # We store per-sample: MSI coords, edge_index, and a list of noise MZ value arrays.
        self.noise_by_sample = {}
        target_mz_set = set(TARGET_MZS)
        n_noise_total = 0
        for sample_id in samples:
            if sample_id not in msi_data:
                continue
            msi = msi_data[sample_id]
            noise_mz_names = [n for n in msi.var_names if n not in target_mz_set]
            if len(noise_mz_names) == 0:
                continue
            msi_coords = np.column_stack([
                msi.obs['x_um'].values.astype(float),
                msi.obs['y_um'].values.astype(float)
            ])
            msi_edge_index = build_knn_graph(msi_coords, config['k_neighbors'])
            noise_values_list = []
            for mz_name in noise_mz_names:
                _, values = extract_pattern_data(msi, mz_name)
                if values is not None:
                    noise_values_list.append(values.copy())
            self.noise_by_sample[sample_id] = {
                'coords': msi_coords.copy(),
                'edge_index': msi_edge_index,
                'noise_values': noise_values_list,
            }
            n_noise_total += len(noise_values_list)
        print(f"  Noise: {n_noise_total} noise MZ features across {len(self.noise_by_sample)} samples", flush=True)

    def __len__(self):
        return len(self.pairs)

    def get_batch(self, batch_size, augment=True):
        indices = np.random.choice(len(self.pairs), batch_size, replace=True)
        rna_graphs, msi_graphs, meta = [], [], []
        for idx in indices:
            pair = self.pairs[idx]
            rc, rv, mc, mv = self.augmentor.augment_pair(
                pair['rna_coords'].copy(), pair['rna_values'].copy(),
                pair['msi_coords'].copy(), pair['msi_values'].copy(),
                augment=augment
            )
            rna_graphs.append(Data(x=make_features(rc, rv), edge_index=pair['rna_edge_index']))
            msi_graphs.append(Data(x=make_features(mc, mv), edge_index=pair['msi_edge_index']))
            meta.append({'gene': pair['gene'], 'mz': pair['mz'], 'sample_id': pair['sample_id']})
        return rna_graphs, msi_graphs, meta

    def get_noise_batch(self, n_noise):
        """Sample n_noise random noise MZ graphs from random samples."""
        if not self.noise_by_sample:
            return []
        sample_ids = list(self.noise_by_sample.keys())
        noise_graphs = []
        for _ in range(n_noise):
            sid = sample_ids[np.random.randint(len(sample_ids))]
            noise_info = self.noise_by_sample[sid]
            vals = noise_info['noise_values']
            chosen_vals = vals[np.random.randint(len(vals))]
            coords = noise_info['coords'].copy()
            values = chosen_vals.copy()
            coords = self.augmentor.apply_geometric_transform(coords)
            noise_level = self.config.get('aug_msi_noise', 0.05)
            if noise_level > 0:
                n = np.random.normal(0, noise_level * values.std(), len(values))
                values = values + n
            noise_graphs.append(Data(
                x=make_features(coords, values),
                edge_index=noise_info['edge_index']
            ))
        return noise_graphs


def encode_noise_subbatch(model, noise_graphs, sub_batch_size, device):
    """Encode noise graphs in sub-batches with no_grad to save GPU memory.
    
    Noise embeddings are detached (no gradient) — similar to MoCo's momentum encoder.
    The MSI encoder still learns from positive pairs; the RNA encoder learns to
    push away from noise via the contrastive loss.
    """
    all_embs = []
    for i in range(0, len(noise_graphs), sub_batch_size):
        sub = noise_graphs[i:i + sub_batch_size]
        with torch.no_grad():
            batch = Batch.from_data_list(sub).to(device)
            h = model.msi_encoder.encode_batch(batch)
            embs = F.normalize(model.msi_projection(h), dim=-1)
            all_embs.append(embs)
            del batch, h  # free GPU memory between sub-batches
    return torch.cat(all_embs, dim=0).detach()  # N x D, no gradient


def train_gnn(train_samples, config, label=""):
    """Train a GNN model on the given samples with noise negatives. Returns (model, losses)."""
    print(f"\n{'='*60}", flush=True)
    print(f"Training GNN {label} on {len(train_samples)} samples...", flush=True)
    print(f"  Samples: {train_samples}", flush=True)

    dataset = PrecomputedDataset(rna_data, msi_data, train_samples, config)
    model = CrossModalGNN(config).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay']
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2)
    loss_fn = InfoNCELoss(temperature=config['temperature'])

    n_noise = config.get('noise_negatives_per_batch', 0)
    noise_sub_bs = config.get('noise_sub_batch_size', 10)
    print(f"  Noise negatives per batch: {n_noise} (sub-batch: {noise_sub_bs})", flush=True)

    losses = []
    best_loss = float('inf')
    best_state = None
    patience_counter = 0
    batches_per_epoch = max(15, len(dataset) // config['batch_size'])
    start_time = time.time()

    for epoch in range(config['epochs']):
        model.train()
        epoch_loss = 0.0
        for _ in range(batches_per_epoch):
            rna_graphs, msi_graphs, _ = dataset.get_batch(config['batch_size'], augment=True)
            rna_batch = Batch.from_data_list(rna_graphs).to(DEVICE)
            msi_batch = Batch.from_data_list(msi_graphs).to(DEVICE)

            rna_all = model.rna_encoder.encode_batch(rna_batch)
            msi_all = model.msi_encoder.encode_batch(msi_batch)
            rna_embs = F.normalize(model.rna_projection(rna_all), dim=-1)
            msi_embs = F.normalize(model.msi_projection(msi_all), dim=-1)

            # Encode noise negatives (no_grad, sub-batched)
            noise_embs = None
            if n_noise > 0:
                noise_graphs = dataset.get_noise_batch(n_noise)
                if len(noise_graphs) > 0:
                    noise_embs = encode_noise_subbatch(
                        model, noise_graphs, noise_sub_bs, DEVICE
                    )

            loss = loss_fn(rna_embs, msi_embs, noise_embs=noise_embs)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        avg_loss = epoch_loss / batches_per_epoch
        losses.append(avg_loss)
        if avg_loss < best_loss - 1e-4:
            best_loss = avg_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        if epoch == 0:
            elapsed = time.time() - start_time
            print(f"  Epoch   1: Loss = {avg_loss:.4f} ({elapsed:.1f}s)", flush=True)
        if (epoch + 1) % 50 == 0:
            elapsed = time.time() - start_time
            print(f"  Epoch {epoch+1:3d}: Loss = {avg_loss:.4f} (best={best_loss:.4f}, {elapsed:.0f}s)", flush=True)
        if patience_counter >= config['patience']:
            print(f"  Early stopping at epoch {epoch+1}", flush=True)
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    total_time = time.time() - start_time
    print(f"  Best loss: {best_loss:.4f}, Time: {total_time:.0f}s ({total_time/60:.1f} min)", flush=True)
    return model, losses


print("Training functions defined.")

Training functions defined.


## GNN Rotation Scoring

Generates GNN cosine similarity scores matching the format of baseline's 0_1 output.

**CRITICAL**: Rotates **RNA** coordinates (not MSI) to match baseline's convention.

In [6]:
# ============================================================================
# GNN ROTATION SCORING
# ============================================================================

def gnn_rotation_scoring(model, samples, angles_deg, config):
    """
    Generate GNN cosine similarity scores for all (gene, sample, angle) combinations.
    
    Convention: RNA is rotated (matching baseline's 0_1), MSI stays fixed.
    Optimization: MSI embeddings pre-computed once (MSI never rotated).
    
    Returns list of dicts with keys:
      Rotation_Angle, Gene, Sample, Best_MZ_GNN_Cosine, Max_GNN_Cosine
    """
    model.eval()
    print(f"\nGNN scoring: {len(samples)} samples x {len(angles_deg)} angles...", flush=True)
    
    all_rows = []
    
    with torch.no_grad():
        for sample_id in samples:
            t0 = time.time()
            rna = rna_data[sample_id]
            msi = msi_data[sample_id]
            
            # 1. Pre-compute ALL MSI embeddings (MSI never rotated)
            mz_embeddings = {}  # mz_name -> embedding vector
            mz_names = list(msi.var_names)  # All 550 features (target + noise)
            for mz_name in mz_names:
                coords, values = extract_pattern_data(msi, mz_name)
                if coords is not None:
                    msi_graph = prepare_graph(coords, values, config['k_neighbors'])
                    msi_graph = msi_graph.to(DEVICE)
                    emb = model.encode_msi(msi_graph.x, msi_graph.edge_index).cpu().numpy()
                    mz_embeddings[mz_name] = emb
            
            # 2. Pre-compute RNA k-NN edge_index (rotation-invariant)
            gene_cache = {}  # gene -> (coords, values, edge_index)
            for gene in TARGET_GENES:
                coords, values = extract_pattern_data(rna, gene)
                if coords is not None:
                    edge_index = build_knn_graph(coords, config['k_neighbors'])
                    gene_cache[gene] = (coords, values, edge_index)
            
            # 3. For each rotation angle, rotate RNA and score
            for angle_deg in angles_deg:
                angle_rad = np.deg2rad(angle_deg)
                cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)
                rot_mat = np.array([[cos_a, -sin_a], [sin_a, cos_a]])
                
                for gene in TARGET_GENES:
                    if gene not in gene_cache:
                        continue
                    coords, values, edge_index = gene_cache[gene]
                    
                    # Rotate RNA coordinates
                    center = coords.mean(axis=0)
                    rotated = (coords - center) @ rot_mat.T + center
                    
                    # Build node features with rotated coords
                    x_feat = make_features(rotated, values)
                    rna_graph = Data(x=x_feat, edge_index=edge_index).to(DEVICE)
                    gene_emb = model.encode_rna(rna_graph.x, rna_graph.edge_index).cpu().numpy()
                    
                    # Cosine similarity with all MSI embeddings
                    scores = {mz: float(np.dot(gene_emb, emb))
                              for mz, emb in mz_embeddings.items()}
                    
                    # Find best MZ
                    best_mz = max(scores, key=scores.get)
                    best_score = scores[best_mz]
                    
                    all_rows.append({
                        'Rotation_Angle': angle_to_csv(angle_deg),
                        'Gene': gene,
                        'Sample': sample_id,
                        'Best_MZ_GNN_Cosine': best_mz,
                        'Max_GNN_Cosine': best_score,
                    })
            
            elapsed = time.time() - t0
            print(f"  {sample_id}: done ({elapsed:.1f}s)", flush=True)
    
    print(f"  Total rows: {len(all_rows)}", flush=True)
    return all_rows


print("Scoring functions defined.")

Scoring functions defined.


## Main Execution

Runs CV and/or full training based on `TRAIN_MODE`, then merges GNN scores with traditional metrics.

In [ ]:
# ============================================================================
# MAIN EXECUTION
# ============================================================================

print(f"\n{'='*70}")
print(f"0_1_5 GNN SCORING - MODE: {TRAIN_MODE}")
print(f"{'='*70}\n", flush=True)

gnn_score_rows = []  # Accumulated GNN scores for all samples

# ---- Part 1: Cross-Validation (for fair scoring) ----
if TRAIN_MODE in ("CV_AND_FULL", "CV_ONLY"):
    print("\n" + "="*60)
    print("PART 1: 4-FOLD CROSS-VALIDATION")
    print("="*60, flush=True)
    
    for fold_id, fold in CV_FOLDS.items():
        print(f"\n--- Fold {fold_id}/4 ---")
        print(f"  Train: {fold['train']}")
        print(f"  Test:  {fold['test']}", flush=True)
        
        # Train on fold's training set
        fold_model, fold_losses = train_gnn(
            fold['train'], CONFIG, label=f"Fold {fold_id}"
        )
        
        # Save fold model
        fold_model_path = os.path.join(OUTPUT_DIR, f"fold{fold_id}_model.pth")
        torch.save(fold_model.state_dict(), fold_model_path)
        print(f"  Saved: {fold_model_path}", flush=True)
        
        # Score the held-out test samples
        fold_rows = gnn_rotation_scoring(
            fold_model, fold['test'], ROTATION_ANGLES_DEG, CONFIG
        )
        gnn_score_rows.extend(fold_rows)
        
        # Quick accuracy check at 0°
        rows_0 = [r for r in fold_rows if r['Rotation_Angle'] == 0]
        correct = sum(1 for r in rows_0 
                      if r['Best_MZ_GNN_Cosine'] == GROUND_TRUTH.get(r['Gene'], ''))
        total = len(rows_0)
        acc = correct / total * 100 if total > 0 else 0
        print(f"  Fold {fold_id} accuracy at 0°: {correct}/{total} = {acc:.1f}%", flush=True)
    
    # Summary
    rows_0_all = [r for r in gnn_score_rows if r['Rotation_Angle'] == 0]
    correct_all = sum(1 for r in rows_0_all
                      if r['Best_MZ_GNN_Cosine'] == GROUND_TRUTH.get(r['Gene'], ''))
    total_all = len(rows_0_all)
    print(f"\n  CV Overall accuracy at 0°: {correct_all}/{total_all} = {correct_all/total_all*100:.1f}%")


# ---- Part 2: Full Model Training ----
if TRAIN_MODE in ("CV_AND_FULL", "FULL_ONLY"):
    print(f"\n{'='*60}")
    print("PART 2: FULL MODEL TRAINING (all 16 samples)")
    print("="*60, flush=True)
    
    full_model, full_losses = train_gnn(
        ALL_SAMPLES, CONFIG, label="Full (all 16)"
    )
    torch.save(full_model.state_dict(), FINAL_MODEL_PATH)
    print(f"  Saved final model: {FINAL_MODEL_PATH}", flush=True)


# ---- Part 3: Inference Mode ----
if TRAIN_MODE == "INFERENCE":
    print(f"\n{'='*60}")
    print("INFERENCE MODE: Loading existing model")
    print("="*60, flush=True)
    
    assert os.path.exists(FINAL_MODEL_PATH), f"Model not found: {FINAL_MODEL_PATH}"
    inf_model = CrossModalGNN(CONFIG).to(DEVICE)
    inf_model.load_state_dict(torch.load(FINAL_MODEL_PATH, map_location=DEVICE))
    inf_model.eval()
    print(f"  Loaded: {FINAL_MODEL_PATH}", flush=True)
    
    # Score all samples with loaded model
    gnn_score_rows = gnn_rotation_scoring(
        inf_model, ALL_SAMPLES, ROTATION_ANGLES_DEG, CONFIG
    )


# ---- Part 4: FULL_ONLY scoring (use full model for all samples) ----
if TRAIN_MODE == "FULL_ONLY":
    print("\nScoring all samples with full model...", flush=True)
    gnn_score_rows = gnn_rotation_scoring(
        full_model, ALL_SAMPLES, ROTATION_ANGLES_DEG, CONFIG
    )


# ---- Part 5: Merge with traditional metrics CSV ----
if len(gnn_score_rows) > 0:
    print(f"\n{'='*60}")
    print("MERGING GNN SCORES WITH TRADITIONAL METRICS")
    print("="*60, flush=True)
    
    # Load traditional CSV
    trad_df = pd.read_csv(INPUT_CSV)
    print(f"  Traditional CSV: {len(trad_df)} rows, columns: {list(trad_df.columns)}")
    
    # Create GNN dataframe
    gnn_df = pd.DataFrame(gnn_score_rows)
    print(f"  GNN scores: {len(gnn_df)} rows")
    
    # Merge on (Rotation_Angle, Gene, Sample)
    merged_df = pd.merge(
        trad_df, gnn_df,
        on=['Rotation_Angle', 'Gene', 'Sample'],
        how='left'  # Keep all traditional rows, add GNN where available
    )
    
    # Verify merge
    n_with_gnn = merged_df['Best_MZ_GNN_Cosine'].notna().sum()
    n_without_gnn = merged_df['Best_MZ_GNN_Cosine'].isna().sum()
    print(f"  Merged: {len(merged_df)} rows ({n_with_gnn} with GNN, {n_without_gnn} without)")
    
    if n_without_gnn > 0:
        print(f"  WARNING: {n_without_gnn} rows missing GNN scores (NaN)")
        missing_samples = merged_df[merged_df['Best_MZ_GNN_Cosine'].isna()]['Sample'].unique()
        print(f"  Missing samples: {list(missing_samples)}")
    
    # Save merged CSV
    merged_df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n  Saved: {OUTPUT_CSV}")
    print(f"  Columns: {list(merged_df.columns)}")
    
    # Quick accuracy summary
    print(f"\n  GNN accuracy summary (from merged CSV):")
    for angle in [0, 6, 354]:  # 0°, +6°, -6° (354=360-6)
        df_a = merged_df[(merged_df['Rotation_Angle'] == angle) & merged_df['Best_MZ_GNN_Cosine'].notna()]
        if len(df_a) > 0:
            correct = sum(df_a.apply(
                lambda r: r['Best_MZ_GNN_Cosine'] == r['Gene'].replace('Gene_', 'MZ_'), axis=1))
            print(f"    Angle {angle}°: {correct}/{len(df_a)} = {correct/len(df_a)*100:.1f}%")
else:
    print("\nNo GNN scores generated. Check TRAIN_MODE setting.")

print(f"\n{'='*70}")
print("DONE")
print(f"{'='*70}")


0_1_5 GNN SCORING - MODE: CV_AND_FULL


PART 1: 4-FOLD CROSS-VALIDATION

--- Fold 1/4 ---
  Train: ['YC_2', 'YC_3', 'YC_4', 'AC_2', 'AC_3', 'AC_4', 'YAD_2', 'YAD_3', 'YAD_4', 'AAD_2', 'AAD_3', 'AAD_4']
  Test:  ['YC_1', 'AC_1', 'YAD_1', 'AAD_1']

Training GNN Fold 1 on 12 samples...
  Samples: ['YC_2', 'YC_3', 'YC_4', 'AC_2', 'AC_3', 'AC_4', 'YAD_2', 'YAD_3', 'YAD_4', 'AAD_2', 'AAD_3', 'AAD_4']
    Pre-computed 100 graph pairs...
